### **1️⃣ PromptTemplate Methods & Options**

#### **a. format**

**When to use:**
- Single-use, dynamic variable replacement.
- Quick one-off prompt generation.

**Important:** Only works with string-based templates, not chat messages.


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Explain {topic} in simple terms.")

result = prompt.format(topic="LangChain")
print(result)

#### **b. partial**

**When to use:**
- Pre-fill some variables (like API keys, company name, tenant ID).
- Great for multi-tenant apps or standardizing system info.
- **Tip:** Combine with chat templates for system messages.


In [ ]:
partial_prompt = prompt.partial(topic="LangChain")
print(partial_prompt.format())
# Output: "Explain LangChain in simple terms."

#### **c. format_prompt**

**What it does:**
- Returns a PromptValue object that keeps metadata.
- Needed if you want to pass structured prompts into LLMs or ChatPromptTemplate.

**When to use:**
- Production pipelines with memory, logging, or multi-step chains.


In [ ]:
prompt_obj = prompt.format_prompt(topic="LangChain")
print(prompt_obj.to_messages())

#### **d. to_messages (ChatPromptTemplate only)**

**When to use:**
- Needed for chat-based models (GPT-4, Claude, etc.)
- Required if using memory or conversation placeholders

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant."),
    ("user", "Explain {topic}.")
])

messages = chat_prompt.format_prompt(topic="LangChain").to_messages()

# Returns a list of dicts like [{"role": "system", "content": ...}, ...]

#### **e. partial with ChatPromptTemplate**

**Use case:**
- Pre-fill variables for system/user messages
- Avoid repeating the same variable in multiple places


In [ ]:
partial_chat = chat_prompt.partial(topic="LangChain")
messages = partial_chat.format_prompt().to_messages()

### **2️⃣ FewShotPromptTemplate Options**

- **example_prompt:** define the format of a single example
- **examples:** list of actual examples
- **prefix & suffix:** text before/after examples
- **input_variables:** variables you’ll provide dynamically

**When to use:**
- Classification, structured generation, or pattern learning
- Can be combined with with_structured_output() for JSON enforcement


In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template("Translate {word} to French: {translation}")

few_shot = FewShotPromptTemplate(
    examples=[{"word": "cat", "translation": "chat"}, {"word": "dog", "translation": "chien"}],
    example_prompt=example_prompt,
    prefix="Here are some translations:",
    suffix="Translate {word} to French:",
    input_variables=["word"]
)

### **3️⃣ Structured Output Options**

**When to use:**
- Always in production if you need guaranteed fields / JSON
- Better than using regex on model outputs
- Can be combined with FewShotPromptTemplate or ChatPromptTemplate.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

class TranslationSchema(BaseModel):
    french_word: str

parser = PydanticOutputParser(pydantic_object=TranslationSchema)
output = parser.parse("chat")  # Returns {"french_word": "chat"}

#### 4️⃣ **Quick Decision Table: Methods in Production**

| Template | Method | When to Use |
|----------|--------|------------|
| `PromptTemplate` | `format` | One-off prompts |
| `PromptTemplate` | `partial` | Pre-fill static variables |
| `PromptTemplate` | `format_prompt` | Multi-step pipelines / logging |
| `ChatPromptTemplate` | `format_prompt().to_messages()` | For chat models / multi-turn / memory |
| `ChatPromptTemplate` | `partial` | Pre-fill system or user variables |
| `FewShotPromptTemplate` | `examples` | Pattern or few-shot tasks |
| Any | Structured Output (`with_structured_output()`) | Production-grade JSON/API output |

---

#### ✅ **Rule of Thumb for Production:**

- Always use `partial` for variables that don’t change per call (system instructions, tenant ID).  
- Always use `format_prompt().to_messages()` for chat models.  
- Prefer structured output over regex or string parsing.  
- Use `FewShotPromptTemplate` only if the model needs examples to follow a format.